# RiskSequencer — 02 Modeling

Baseline vs. sequence model (build-plan Phase 2). Establishes the **LightGBM
baseline** on last-transaction features, trains the **2-layer LSTM + attention**
on the full sequences, and compares them on the held-out test set.

Success criteria from the plan:
- LightGBM baseline AUC > 0.88
- LSTM test AUC ≥ 0.94
- **LSTM beats LightGBM by ≥ 4 AUC points**

> Note: on the *synthetic* data both models saturate (fraud is deliberately
> separable), so the 4-point gap is best evaluated on the real IEEE-CIS data.

### ⚠️ OpenMP guard — run this cell **first**, before any other import

PyTorch and LightGBM each ship their own OpenMP runtime; loading both in one
process segfaults on macOS. Setting these env vars (and single-threading both
libraries below) makes them coexist. Must run before `torch`/`lightgbm` are
imported, so keep it as the very first cell.

In [ ]:
import os
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")

In [ ]:
import sys
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, precision_recall_curve

import torch
torch.set_num_threads(1)            # single OpenMP thread — see guard cell above

from config import RAW_DIR, TrainConfig
from features.feature_pipeline import build_features
from data.sequence_builder import build_sequences, fit_scaler, time_based_split
from models.lgbm_baseline import last_txn_table, train_lgbm
from training.train import run_training, _scores
from training.evaluate import evaluate

## 1. Data and time-based splits

In [ ]:
parquet = RAW_DIR / "transactions.parquet"
if parquet.exists():
    raw = pd.read_parquet(parquet); source = "IEEE-CIS (real)"
else:
    from data.synthetic import generate_transactions
    raw = generate_transactions(n_users=3000, seed=42); source = "synthetic"
print("source:", source)

feats = build_features(raw)
tr, va, te = time_based_split(feats)
print("rows  train/val/test:", len(tr), len(va), len(te))

## 2. LightGBM baseline (last-transaction features)

In [ ]:
Xtr, ytr = last_txn_table(tr)
Xva, yva = last_txn_table(va)
Xte, yte = last_txn_table(te)

lgbm = train_lgbm(Xtr, ytr, Xva, yva, params={"num_threads": 1})  # single-thread (OpenMP guard)
lgbm_test_scores = lgbm.predict(Xte)
lgbm_res = evaluate(yte, lgbm_test_scores)
print(f"LightGBM  test AUC = {lgbm_res.auc_roc:.4f}  (target > 0.88)")

### 2b. SHAP explainability for the baseline

Top feature contributors (skipped automatically if `shap` isn't installed).

In [ ]:
try:
    import shap
    from models.lgbm_baseline import shap_summary
    sv = shap_summary(lgbm, Xva)
    from config import FEATURE_COLUMNS
    shap.summary_plot(sv, features=Xva, feature_names=FEATURE_COLUMNS, show=True)
except Exception as e:
    print("SHAP step skipped:", type(e).__name__, e)

## 3. LSTM + attention (full sequences)

In [ ]:
scaler = fit_scaler(tr)
train_ds = build_sequences(tr, scaler)
val_ds   = build_sequences(va, scaler)
test_ds  = build_sequences(te, scaler)

out = run_training(train_ds, val_ds, TrainConfig(max_epochs=15))
lstm_test_scores = _scores(out.model, test_ds, device="cpu")
lstm_res = evaluate(test_ds.y, lstm_test_scores)
print(f"LSTM      test AUC = {lstm_res.auc_roc:.4f}  (target >= 0.94)")

## 4. Head-to-head comparison

In [ ]:
gap = lstm_res.auc_roc - lgbm_res.auc_roc
print(f"LightGBM test AUC : {lgbm_res.auc_roc:.4f}")
print(f"LSTM     test AUC : {lstm_res.auc_roc:.4f}")
print(f"gap (LSTM - LGBM) : {gap:+.4f}  ({'>= 4 pts ✅' if gap >= 0.04 else 'below 4 pts'})")

# ROC + PR overlays
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
for name, sc in [("LightGBM", lgbm_test_scores), ("LSTM", lstm_test_scores)]:
    fpr, tpr, _ = roc_curve(test_ds.y if name=="LSTM" else yte, sc)
    ax[0].plot(fpr, tpr, label=f"{name}")
    pr, rc, _ = precision_recall_curve(test_ds.y if name=="LSTM" else yte, sc)
    ax[1].plot(rc, pr, label=f"{name}")
ax[0].plot([0,1],[0,1],"k--",alpha=.3); ax[0].set_title("ROC"); ax[0].set_xlabel("FPR"); ax[0].set_ylabel("TPR"); ax[0].legend()
ax[1].set_title("Precision-Recall"); ax[1].set_xlabel("recall"); ax[1].set_ylabel("precision"); ax[1].legend()
plt.tight_layout(); plt.show()

## 5. Findings

> Fill in after running on real IEEE-CIS data.

- LightGBM baseline AUC = ____ (target > 0.88).
- LSTM test AUC = ____ (target ≥ 0.94).
- Gap = ____ pts — the sequence model's advantage comes from modeling the
  *order* of behavior (device change → burst), which the last-transaction
  baseline can't see.
- Log all three runs to MLflow (`risksequencer/lstm-experiments`) and pick the
  operating threshold via `04_error_analysis.ipynb`.
- If the gap is < 4 pts on real data: revisit sequence length, attention, or
  try a bidirectional LSTM (see the plan's risk table).